# Clase 177 — Effect size: Cohen's d, Hedges' g, Cliff's delta

p-value dice si hay diferencia; effect size dice **cuán grande**. Con n grande, p<0.001 con d=0.05 es trivial.
Requiere: `pip install numpy scipy statsmodels pingouin`.

## 🧠 Intuición previa
<!--SOL175184-->

No basta con que la diferencia sea *significativa*: ¿es **grande**? Un p-value chico solo dice "probablemente no es cero"; con muestras enormes hasta un efecto ridiculo cruza `p<0.05`. El **tamaño del efecto** (Cohen's d, Hedges' g, Cliff's δ) mide *cuánto* difieren los grupos en unidades interpretables — el verdadero titular del resultado.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
# Dos grupos con diferencia de medias controlada
n1, n2 = 60, 60
x1 = rng.normal(loc=100, scale=15, size=n1)
x2 = rng.normal(loc=108, scale=15, size=n2)  # diff = 8, sd = 15 -> d ~ 0.53
print(f'mean1={x1.mean():.2f}  mean2={x2.mean():.2f}  diff={x2.mean()-x1.mean():.2f}')

## Cohen's d desde scratch
$d = \dfrac{\bar{x}_1 - \bar{x}_2}{s_p}$, con $s_p$ = pooled SD. Reglas: 0.2 small, 0.5 medium, 0.8 large.

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    sp = np.sqrt(((na-1)*va + (nb-1)*vb) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

d = cohens_d(x2, x1)
print(f"Cohen's d = {d:.3f}")
def interp(d):
    ad = abs(d)
    return 'trivial' if ad<0.2 else 'small' if ad<0.5 else 'medium' if ad<0.8 else 'large'
print('interpretacion:', interp(d))

## Hedges' g — corrección por bias (muestras chicas)
$g = d \cdot \left(1 - \dfrac{3}{4(n_1+n_2)-9}\right)$

In [ ]:
def hedges_g(a, b):
    d = cohens_d(a, b)
    n = len(a) + len(b)
    return d * (1 - 3 / (4*n - 9))

g = hedges_g(x2, x1)
print(f"Hedges' g = {g:.3f}  (d={d:.3f}, casi igual con n={n1+n2})")
# Con muestras chicas la diferencia es mayor:
xa = rng.normal(0, 1, 10); xb = rng.normal(0.7, 1, 10)
print(f'n=20: d={cohens_d(xa,xb):.3f}  g={hedges_g(xa,xb):.3f}')

## Cliff's delta — no paramétrico
$\delta = \dfrac{\#(x_i > y_j) - \#(x_i < y_j)}{n_1 n_2}$. Rango [-1, 1]. Robusto a outliers.

In [ ]:
def cliffs_delta(a, b):
    a = np.asarray(a); b = np.asarray(b)
    diff = a[:, None] - b[None, :]
    return (np.sum(diff > 0) - np.sum(diff < 0)) / (len(a) * len(b))

delta = cliffs_delta(x2, x1)
print(f"Cliff's delta = {delta:.3f}")
# Umbrales (Romano): |d|<0.147 negligible, <0.33 small, <0.474 medium, else large
ad = abs(delta)
print('interp:', 'negligible' if ad<0.147 else 'small' if ad<0.33 else 'medium' if ad<0.474 else 'large')

## Pingouin: validación cruzada

In [ ]:
try:
    import pingouin as pg
    print('pingouin d  =', pg.compute_effsize(x2, x1, eftype='cohen'))
    print('pingouin g  =', pg.compute_effsize(x2, x1, eftype='hedges'))
    print('pingouin CL =', pg.compute_effsize(x2, x1, eftype='CLES'))  # common language effect size
    print('manual   d  =', d)
except ImportError:
    print('pingouin no instalado; usar `pip install pingouin`')

## Demo: p<0.001 puede ser trivial
Con n=100000 una diferencia mínima es 'significativa' pero d ~ 0.05 (irrelevante en la práctica).

In [ ]:
rng2 = np.random.default_rng(42)
big_a = rng2.normal(100, 15, 100_000)
big_b = rng2.normal(100.75, 15, 100_000)
t, p = stats.ttest_ind(big_a, big_b)
print(f'p-value = {p:.3e}  (super significativo)')
print(f"Cohen's d = {cohens_d(big_b, big_a):.3f}  (trivial)")
print('Lección: con n enorme, todo es significativo. Reportá SIEMPRE effect size + CI.')

## Power analysis
¿Qué n necesito para detectar d=0.3 con power=0.80, alpha=0.05?

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
n_needed = analysis.solve_power(effect_size=0.3, alpha=0.05, power=0.80, alternative='two-sided')
print(f'n por grupo para d=0.3, power=0.80 -> {n_needed:.0f}')
# Y al revés: con n=60 por grupo y d=0.5, ¿qué power tengo?
p_obs = analysis.power(effect_size=0.5, nobs1=60, alpha=0.05, ratio=1.0)
print(f'power con n=60/grupo y d=0.5 -> {p_obs:.3f}')

## Takeaways
1. `p-value` responde *¿hay diferencia?*; effect size responde *¿cuánta?*.
2. Cohen's d (paramétrico, normal), Hedges' g (corrige bias con n chico), Cliff's delta (no paramétrico, robusto).
3. Reportá siempre **effect size + CI**, no solo p.
4. Power analysis ANTES de correr el experimento — evita estudios subpowered.

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Cohen's d a mano** para `tip` por `sex`, con `s_pooled`.

In [ ]:
# <!--SOL175184-->
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(42)
n = 244
sex = rng.choice(["Male", "Female"], size=n, p=[0.64, 0.36])
tip = np.clip(rng.lognormal(2.85, 0.38, n)*0.15 + rng.normal(0, 0.5, n) + np.where(sex=="Male", 0.15, 0.0), 1.0, None)
a = tip[sex == "Male"]; b = tip[sex == "Female"]
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp
d = cohens_d(a, b)
print(f"Cohen's d = {d:.4f}")
assert np.isfinite(d)
print("Cohen's d manual: OK")

<!--SOL175184-->

**Ej. 2 — Varios `eftype`** (cohen/hedges/glass/CLES) desde cero; si `pingouin` está, se cruza.

In [ ]:
# <!--SOL175184-->
def hedges_g(a, b):
    na, nb = len(a), len(b)
    J = 1 - 3 / (4*(na+nb) - 9)
    return J * cohens_d(a, b)
def glass_delta(a, b):
    return (a.mean() - b.mean()) / b.var(ddof=1)**0.5
def cles(a, b):
    diff = a[:, None] - b[None, :]
    return (diff > 0).mean() + 0.5*(diff == 0).mean()
print(f"cohen={cohens_d(a,b):.4f}  hedges={hedges_g(a,b):.4f}  glass={glass_delta(a,b):.4f}  CLES={cles(a,b):.4f}")
try:
    import pingouin as pg
    for ef in ("cohen", "hedges", "glass", "CLES"):
        print(f"[pingouin] {ef}: {pg.compute_effsize(a, b, eftype=ef):.4f}")
except ModuleNotFoundError:
    print("(pingouin no instalado -> validacion cruzada omitida)")
assert 0 <= cles(a, b) <= 1
print("Cuatro effect sizes: OK")

<!--SOL175184-->

**Ej. 3 — Hedges' g con `n=10`.** `|g| < |d|` (corrige el sesgo al alza).

In [ ]:
# <!--SOL175184-->
a10 = rng.normal(1.0, 1.0, 10)
b10 = rng.normal(0.3, 1.0, 10)
d10, g10 = cohens_d(a10, b10), hedges_g(a10, b10)
print(f"d={d10:.4f}  g={g10:.4f}  g/d={g10/d10:.3f}")
assert abs(g10) < abs(d10)
print("Hedges' g < Cohen's d en n chico: OK")

<!--SOL175184-->

**Ej. 4 — Cliff's δ** para datos ordinales/asimétricos (robusto).

In [ ]:
# <!--SOL175184-->
def cliffs_delta(a, b):
    diff = a[:, None] - b[None, :]
    return np.sign(diff).mean()
likert_a = rng.integers(1, 6, 40)
likert_b = rng.integers(1, 6, 40) + 1
delta = cliffs_delta(likert_a, likert_b)
mag = ("trivial", "pequeno", "medio", "grande")[int(np.searchsorted([.147, .33, .474], abs(delta)))]
print(f"Cliff's delta = {delta:.4f}  ({mag})")
assert -1 <= delta <= 1 and delta < 0
print("Cliff's delta interpretado: OK")

<!--SOL175184-->

**Ej. 5 — Effect size + IC.** Bootstrap del Cohen's d → IC95 %.

In [ ]:
# <!--SOL175184-->
B = 5_000
na, nb = len(a), len(b)
boot = np.array([cohens_d(rng.choice(a, na, replace=True), rng.choice(b, nb, replace=True)) for _ in range(B)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"d={d:.3f}  IC95% bootstrap=[{lo:.3f},{hi:.3f}]")
assert lo <= d <= hi
print("IC bootstrap contiene al d puntual: OK")